# 05 · Análisis de lenguaje

## 1. Tokenización

Esta sección separa los resúmenes en palabras y números para preparar las representaciones de texto. Parte de las copias limpias del notebook 01 y conserva los identificadores, el texto y las puntuaciones. La misma regla se aplica a los textos de origen para permitir comparaciones posteriores.

La tokenización convierte el texto a minúsculas y unifica las formas Unicode equivalentes. Conserva las contracciones y los posesivos con apóstrofo dentro de una palabra, como don't y student's. Separa las palabras con guion, elimina la puntuación que sirve de separador y conserva los números. Un decimal queda separado por el punto. Esta regla sencilla no distingue todos los usos lingüísticos de los apóstrofos.

No se eliminan palabras frecuentes, no se corrige la ortografía y no se reducen palabras a su raíz. Así se conserva el orden de las palabras y las negaciones para las siguientes técnicas. El texto original se mantiene en una columna aparte. La tokenización usa expresiones regulares y no necesita descargar modelos.

### 1.1. Carga de datos limpios

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_csv_files

INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
for directory in [TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

clean_files = ["summaries_train_clean.csv", "prompts_train_clean.csv"]
if not all((INTERIM_DATA_DIR / name).is_file() for name in clean_files):
    raise FileNotFoundError(
        "Faltan las copias limpias. Ejecute primero 00_data_loading.ipynb "
        "y 01_data_quality.ipynb con los CSV de entrenamiento en data/raw/."
    )
datasets = load_csv_files(INTERIM_DATA_DIR, clean_files)
summaries = datasets["summaries_train_clean"]
prompts = datasets["prompts_train_clean"]
assert not summaries.empty, "No hay resúmenes para analizar."
assert summaries["student_id"].notna().all() and summaries["student_id"].is_unique
assert prompts["prompt_id"].notna().all() and prompts["prompt_id"].is_unique
assert summaries["prompt_id"].notna().all()
assert summaries["prompt_id"].isin(prompts["prompt_id"]).all()
plt.rcParams.update({"figure.figsize": [10, 4], "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

In [2]:
from src.tokenization import tokenize_text

for frame, columns in [(summaries, ["text"]), (prompts, ["prompt_text"])]:
    for column in columns:
        assert frame[column].map(lambda value: isinstance(value, str)).all(), "Hay textos faltantes o no válidos."
        assert frame[column].str.strip().ne("").all(), "Hay textos vacíos."

tokenized_summaries = summaries.copy()
tokenized_prompts = prompts.copy()
tokenized_summaries["tokens"] = summaries["text"].map(tokenize_text)
tokenized_prompts["tokens"] = prompts["prompt_text"].map(tokenize_text)
tokenized_summaries["token_count"] = tokenized_summaries["tokens"].str.len()
tokenized_prompts["token_count"] = tokenized_prompts["tokens"].str.len()

assert tokenized_summaries[summaries.columns].equals(summaries)
assert tokenized_prompts[prompts.columns].equals(prompts)
with pd.option_context("display.max_colwidth", 140):
    display(tokenized_summaries[["student_id", "text", "tokens", "token_count"]].head())
    display(tokenized_prompts[["prompt_id", "prompt_title", "token_count"]])

,student_id,text,tokens,token_count
0,000e8c3c7ddb,The third wave was an experimentto see how people reacted to a new one leader government. It gained popularity as people wanted to try n...,"[the, third, wave, was, an, experimentto, see, how, people, reacted, to, a, new, one, leader, government, it, gained, popularity, as, pe...",61
1,0020ae56ffbf,They would rub it up with soda to make the smell go away and it wouldnt be a bad smell. Some of the meat would be tossed on the floor wh...,"[they, would, rub, it, up, with, soda, to, make, the, smell, go, away, and, it, wouldnt, be, a, bad, smell, some, of, the, meat, would, ...",52
2,004e978e639e,"In Egypt, there were many occupations and social classes involved in day-to-day living. In many instances if you were at the bottom of t...","[in, egypt, there, were, many, occupations, and, social, classes, involved, in, day, to, day, living, in, many, instances, if, you, were...",237
3,005ab0199905,The highest class was Pharaohs these people were gods.Then the 2nd highest class was a gonvener.Chiefs minister were called a vizier as ...,"[the, highest, class, was, pharaohs, these, people, were, gods, then, the, 2nd, highest, class, was, a, gonvener, chiefs, minister, were...",28
4,0070c9e7af47,"The Third Wave developed rapidly because the students genuinly believed that it was the best course of action. Their grades, acomplishm...","[the, third, wave, developed, rapidly, because, the, students, genuinly, believed, that, it, was, the, best, course, of, action, their, ...",203


,prompt_id,prompt_title,token_count
0,39c16e,On Tragedy,600
1,3b9047,Egyptian Social Structure,554
2,814d6b,The Third Wave,598
3,ebad26,Excerpt from The Jungle,976


### 1.2. Revisión de la tokenización

Se compara el conteo de tokens con el conteo por espacios que aparece en el notebook 01. Pueden ser diferentes porque aquí la puntuación separa palabras y los símbolos aislados no cuentan como tokens. No se interpreta esa diferencia como un error.

Se revisan los textos que quedan sin tokens. No se eliminan de forma automática. También se muestra el tamaño del vocabulario como una comprobación de la transformación, sin desarrollar todavía las otras representaciones de lenguaje.

In [3]:
tokenized_summaries["whitespace_word_count"] = summaries["text"].str.split().str.len()
tokenized_summaries["token_count_difference"] = (
    tokenized_summaries["token_count"] - tokenized_summaries["whitespace_word_count"]
)
token_quality = pd.DataFrame([
    {"dataset": name, "textos": len(frame), "tokens_totales": int(frame["token_count"].sum()),
     "vocabulario": len({token for tokens in frame["tokens"] for token in tokens}),
     "sin_tokens": int(frame["token_count"].eq(0).sum())}
    for name, frame in [("summaries", tokenized_summaries), ("prompts", tokenized_prompts)]
])
display(token_quality)
token_quality.to_csv(TABLES_DIR / "05_tokenization_quality.csv", index=False)
token_description = tokenized_summaries[["token_count", "whitespace_word_count", "token_count_difference"]].describe().T
display(token_description.round(2))
token_description.to_csv(TABLES_DIR / "05_tokenization_descriptive.csv", index_label="variable")
different = tokenized_summaries["token_count_difference"].ne(0)
print(f"Hay {different.sum():,} resúmenes con un conteo diferente al basado en espacios.")
with pd.option_context("display.max_colwidth", 140):
    display(tokenized_summaries.loc[different, ["text", "tokens", "token_count", "whitespace_word_count"]].head())
for frame, column in [(tokenized_summaries, "text"), (tokenized_prompts, "prompt_text")]:
    if frame["token_count"].eq(0).any():
        display(frame.loc[frame["token_count"].eq(0), ["prompt_id", column]])

,dataset,textos,tokens_totales,vocabulario,sin_tokens
0,summaries,7165,538507,12469,0
1,prompts,4,2728,949,0


,count,mean,std,min,25%,50%,75%,max
token_count,7165.0,75.16,53.79,25.0,39.0,58.0,92.0,660.0
whitespace_word_count,7165.0,74.81,53.50,22.0,39.0,58.0,92.0,647.0
token_count_difference,7165.0,0.35,1.43,-10.0,0.0,0.0,0.0,22.0


Hay 2,411 resúmenes con un conteo diferente al basado en espacios.


,text,tokens,token_count,whitespace_word_count
2,"In Egypt, there were many occupations and social classes involved in day-to-day living. In many instances if you were at the bottom of t...","[in, egypt, there, were, many, occupations, and, social, classes, involved, in, day, to, day, living, in, many, instances, if, you, were...",237,235
3,The highest class was Pharaohs these people were gods.Then the 2nd highest class was a gonvener.Chiefs minister were called a vizier as ...,"[the, highest, class, was, pharaohs, these, people, were, gods, then, the, 2nd, highest, class, was, a, gonvener, chiefs, minister, were...",28,25
6,The Egyptian society is really different from other socicties I have learned about. As stated in the story in paragraph 6-13 it describe...,"[the, egyptian, society, is, really, different, from, other, socicties, i, have, learned, about, as, stated, in, the, story, in, paragra...",80,78
13,"Aristotle states that an ideal tragedy should have a ""complex plan"" (a well-thought-out plot), ""imitate actions which excite pity and fe...","[aristotle, states, that, an, ideal, tragedy, should, have, a, complex, plan, a, well, thought, out, plot, imitate, actions, which, exci...",54,52
17,They would pickle it in an attempt to keep it fresh. They would insert a hot iron that would burn off parts that had spoiled. They woul...,"[they, would, pickle, it, in, an, attempt, to, keep, it, fresh, they, would, insert, a, hot, iron, that, would, burn, off, parts, that, ...",163,158


### 1.3. Exportación para continuar el análisis

Se guardan los resúmenes y los prompts tokenizados en data/processed como JSON Lines. Este formato mantiene cada lista de tokens como una lista y evita convertirla en texto dentro de un CSV. Los archivos se pueden cargar con pd.read_json usando lines=True. Los tokens de los resúmenes quedan disponibles en tokenized_summaries para continuar este notebook.

Estos archivos son derivados y se regeneran al ejecutar esta sección. No se cambia el contenido de data/raw ni de data/interim. Los nombres de las tablas llevan el prefijo 05_tokenization para distinguirlas de las siguientes secciones.

In [4]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename, frame in [
    ("summaries_train_tokenized.jsonl", tokenized_summaries),
    ("prompts_train_tokenized.jsonl", tokenized_prompts),
]:
    path = PROCESSED_DATA_DIR / filename
    frame.to_json(path, orient="records", lines=True, force_ascii=False, double_precision=15)
    restored = pd.read_json(path, orient="records", lines=True, dtype=False, convert_dates=False)
    assert restored["tokens"].tolist() == frame["tokens"].tolist()
    assert restored["prompt_id"].tolist() == frame["prompt_id"].tolist()
    if "student_id" in frame:
        assert restored["student_id"].tolist() == frame["student_id"].tolist()
    print(f"Guardado: data/processed/{filename}, con {len(frame):,} filas.")

Guardado: data/processed/summaries_train_tokenized.jsonl, con 7,165 filas.
Guardado: data/processed/prompts_train_tokenized.jsonl, con 4 filas.


### 1.4. Resultado de la preparación

La tokenización mantiene la relación entre cada resumen, su prompt y sus puntuaciones. Las listas conservan el orden de las palabras y quedan listas para formar secuencias o matrices de texto en las siguientes secciones. Las diferencias frente al conteo por espacios dependen de las reglas usadas y deben considerarse al comparar longitudes con otros notebooks.

In [5]:
summary_quality = token_quality.loc[token_quality["dataset"].eq("summaries")].iloc[0]
print(f"Se prepararon {summary_quality['textos']:,} resúmenes con "
      f"{summary_quality['tokens_totales']:,} tokens y "
      f"{summary_quality['vocabulario']:,} tokens distintos.")
print(f"La mediana es de {tokenized_summaries['token_count'].median():.1f} tokens por resumen.")
print(f"Hay {summary_quality['sin_tokens']:,} resúmenes sin tokens después de aplicar las reglas.")

Se prepararon 7,165 resúmenes con 538,507 tokens y 12,469 tokens distintos.
La mediana es de 58.0 tokens por resumen.
Hay 0 resúmenes sin tokens después de aplicar las reglas.


### Hallazgos de la tokenización

Los 7,165 resúmenes producen 538,507 tokens y un vocabulario de 12,469 tokens distintos. La mediana es de 58 tokens por resumen. Ningún resumen queda sin tokens, por lo que esta transformación permite mantener todas las observaciones.

En 2,411 resúmenes el conteo cambia frente al conteo por espacios del notebook 01. La diferencia es esperable por el tratamiento de signos y separadores. Para las siguientes representaciones conviene usar las mismas listas de tokens y mantener explícita la regla de conteo al comparar longitudes.

Los archivos procesados incluyen también los cuatro textos de origen tokenizados. Las listas conservan el orden de las palabras y quedan disponibles para continuar el análisis de lenguaje.